# HDFC Bank: Fraud Risk Analysis - Part 0

**Objective:** Data Understanding and Schema Review

Reviewing the latest transaction extract from Data Engineering. 

### Data Dictionary

**Core Transaction Features**
- `TransactionID`: Unique identifier for the transaction.
- `isFraud`: Target variable (1 = Fraud, 0 = Legitimate).
- `TransactionDT`: Time delta from a hidden reference date (in seconds).
- `TransactionAmt`: Payment amount in USD.
- `ProductCD`: Product code.

**Card & Address Features**
- `card1` - `card6`: Payment card information (e.g., Visa/Mastercard, Credit/Debit).
- `addr1`, `addr2`: Purchaser billing region and country.
- `dist1`, `dist2`: Distances.

**Email Domains**
- `P_emaildomain`, `R_emaildomain`: Purchaser and recipient emails.

**Engineered Features (Payment Processor)**
- `C1` - `C14`: Counting features.
- `D1` - `D15`: Time delta features.
- `M1` - `M9`: Match features.
- `V1` - `V339`: Vesta engineered rich features (anonymized).

**Identity Features (Often Missing)**
- `id_01` - `id_38`: Network connection and digital identity.
- `DeviceType`: e.g., mobile, desktop.
- `DeviceInfo`: e.g., Windows, iOS.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))
from fraudguard.data.ingestion import load_bank_data

data_dir = Path.cwd().parent / "data" / "raw"
print("Loading HDFC transaction data warehouse extract...")
df = load_bank_data(data_dir)
print(f"\nData loaded successfully! Shape: {df.shape}")

Loading HDFC transaction data warehouse extract...

Data loaded successfully! Shape: (590540, 434)


### 1. Basic Inspection

In [2]:
pd.set_option('display.max_columns', 50)
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,...,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,19.0,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0,87.0,287.0,NaN,outlook.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,87.0,NaN,NaN,yahoo.com,NaN,2.0,5.0,0.0,0.0,0.0,4.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,...,NotFound,166.0,NaN,542.0,144.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,Android 7.0,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


### 2. Data Types and Memory Usage

In [3]:
print("Dataset Memory Usage and Types:")
df.info(memory_usage='deep')

categorical_cols = df.select_dtypes(include=['object', 'category']).columns
numerical_cols = df.select_dtypes(include=[np.number]).columns

print(f"\nTotal Categorical Features: {len(categorical_cols)}")
print(f"Total Numerical Features: {len(numerical_cols)}")

Dataset Memory Usage and Types:
<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 434 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(4), str(31)
memory usage: 2.5 GB

Total Categorical Features: 31
Total Numerical Features: 403


### 3. Summary Statistics

In [4]:
# Numerical features summary (showing a subset for readability)
df[['TransactionAmt', 'TransactionDT', 'isFraud', 'card1', 'C1', 'D1', 'V1']].describe()

,TransactionAmt,TransactionDT,isFraud,card1,C1,D1,V1
count,590540.000000,5.905400e+05,590540.000000,590540.000000,590540.000000,589271.000000,311253.000000
mean,135.027176,7.372311e+06,0.034990,9898.734658,14.092458,94.347568,0.999945
std,239.162522,4.617224e+06,0.183755,4901.170153,133.569018,157.660387,0.007390
min,0.251000,8.640000e+04,0.000000,1000.000000,0.000000,0.000000,0.000000
25%,43.321000,3.027058e+06,0.000000,6019.000000,1.000000,0.000000,1.000000
50%,68.769000,7.306528e+06,0.000000,9678.000000,1.000000,3.000000,1.000000
75%,125.000000,1.124662e+07,0.000000,14184.000000,3.000000,122.000000,1.000000
max,31937.391000,1.581113e+07,1.000000,18396.000000,4685.000000,640.000000,1.000000


In [5]:
# Categorical features summary
df[categorical_cols].describe()

,ProductCD,card4,card6,P_emaildomain,R_emaildomain,M1,M2,M3,M4,M5,M6,M7,M8,M9,id_12,id_15,id_16,id_23,id_27,id_28,id_29,id_30,id_31,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
count,590540,588963,588969,496084,137291,319440,319440,319440,309096,240058,421180,244275,244288,244288,144233,140985,129340,5169,5169,140978,140978,77565,140282,73289,77805,140985,140985,140985,140985,140810,118666
unique,5,4,4,59,60,2,2,2,3,2,2,2,2,2,2,3,2,3,2,2,2,75,130,260,4,2,2,2,2,2,1786
top,W,visa,debit,gmail.com,gmail.com,T,T,T,M0,F,F,F,F,T,NotFound,Found,Found,IP_PROXY:TRANSPARENT,Found,Found,Found,Windows 10,chrome 63.0,1920x1080,match_status:2,T,F,T,F,desktop,Windows
freq,439670,384767,439938,228355,57147,319415,285468,251731,196405,132491,227856,211374,155251,205656,123025,67728,66324,3489,5155,76232,74926,21155,22000,16874,60011,77814,134066,110452,73922,85165,47722


### 4. Missing Values Analysis

In [6]:
# Top 20 columns with the most missing values
missing_percent = (df.isnull().sum() / len(df)) * 100
print("Top 20 columns with highest missing value percentage:")
print(missing_percent.sort_values(ascending=False).head(20))

Top 20 columns with highest missing value percentage:
id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
id_26    99.125715
id_27    99.124699
id_23    99.124699
id_22    99.124699
dist2    93.628374
D7       93.409930
id_18    92.360721
D13      89.509263
D14      89.469469
D12      89.041047
id_04    88.768923
id_03    88.768923
D6       87.606767
id_33    87.589494
id_09    87.312290
dtype: float64
